# Cost Intelligence Layer — End-to-End Demo

**Phase 2 — Multi-Cloud FinOps Cost Attribution & Intelligence Framework**  
Keerthi Rapolu · April 2026

This notebook walks through the full Phase 2 intelligence pipeline:

1. Load `fct_unified_billing` from DuckDB  
2. Summarise raw cost data by cloud and team  
3. Waste Detection — identify idle, zombie, and misused commitment resources  
4. Waste summary stats and visualisation  
5. Causal Reasoning — explain WHY costs look the way they do  
6. Cost trend visualisation with anomaly markers  
7. Impact Simulation — estimate recoverable savings with risk scores  
8. Executive summary

> **Data note:** All data is synthetic — generated by `load_synthetic.py` and modelled through dbt.  
> Run `make pipeline` (or `make pipeline MONTH=YYYY-MM`) before executing this notebook.

## Cell 1 — Setup

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Make sure repo root is on the path so intelligence/ modules are importable
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

DB_PATH = repo_root / "finops_dbt.duckdb"
assert DB_PATH.exists(), f"DuckDB not found at {DB_PATH}. Run `make pipeline` first."

print(f"DuckDB: {DB_PATH}")
print(f"Repo:   {repo_root}")

In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
df  = con.execute("SELECT * FROM marts.fct_unified_billing").df()
con.close()

print(f"Rows loaded : {len(df):,}")
print(f"Billing months: {sorted(df['billing_month'].unique())}")
print(f"Clouds        : {sorted(df['cloud_provider'].unique())}")
print(f"Teams         : {sorted(df['allocated_team'].dropna().unique())}")
df.head(3)

## Cell 2 — Raw Data Summary

In [ ]:
print("Shape:", df.shape)
print()

by_cloud = (
    df.groupby("cloud_provider")
    .agg(nec_used=("nec_used", "sum"), nec_waste=("nec_waste", "sum"), rows=("nec_used", "count"))
    .reset_index()
)
by_cloud["nec_used"]  = by_cloud["nec_used"].round(2)
by_cloud["nec_waste"] = by_cloud["nec_waste"].round(2)

print("NEC by cloud:")
print(by_cloud.to_string(index=False))
print()

by_team = (
    df.groupby("allocated_team", dropna=False)
    .agg(nec_used=("nec_used", "sum"), nec_waste=("nec_waste", "sum"))
    .reset_index()
    .sort_values("nec_used", ascending=False)
)
by_team["nec_used"]  = by_team["nec_used"].round(2)
by_team["nec_waste"] = by_team["nec_waste"].round(2)

print("NEC by team:")
print(by_team.to_string(index=False))

In [ ]:
by_month_cloud = (
    df.groupby(["billing_month", "cloud_provider"])["nec_used"]
    .sum()
    .reset_index()
)

fig = px.bar(
    by_month_cloud,
    x="billing_month", y="nec_used", color="cloud_provider",
    color_discrete_map={"aws": "#f97316", "azure": "#2563eb", "gcp": "#22c55e"},
    labels={"nec_used": "NEC Used (USD)", "billing_month": "Month", "cloud_provider": "Cloud"},
    title="NEC Used by Cloud and Month",
    barmode="group",
)
fig.update_layout(height=380, legend=dict(orientation="h", y=1.12))
fig.show()

## Cell 3 — Waste Detection

In [ ]:
from intelligence.waste_detector import run as detect_waste

findings = detect_waste(df)

print(f"Waste findings: {len(findings)}")
print()

findings_df = pd.DataFrame(findings)
findings_df[["resource_id", "cloud_provider", "allocated_team",
             "waste_type", "nec_waste", "nec_used", "confidence", "billing_month"]]

## Cell 4 — Waste Summary Stats

In [ ]:
findings_df["cost_impact"] = findings_df.apply(
    lambda r: r["nec_waste"] if r["nec_waste"] > 0 else r["nec_used"], axis=1
)

total_waste   = findings_df["cost_impact"].sum()
total_nec_all = df["nec_used"].sum()
waste_pct     = total_waste / total_nec_all * 100 if total_nec_all else 0

print(f"Total waste detected : ${total_waste:,.2f}")
print(f"Total NEC (all months): ${total_nec_all:,.2f}")
print(f"Waste as % of NEC    : {waste_pct:.2f}%")
print()

_TYPE_LABELS = {
    "unused_commitment":        "Unused Commitment",
    "idle_compute":             "Idle Compute",
    "zombie_resource":          "Zombie Resource",
    "underutilized_commitment": "Underutilised Commitment",
}

by_type = (
    findings_df.groupby("waste_type")["cost_impact"]
    .agg(["sum", "count"])
    .reset_index()
    .rename(columns={"sum": "total_cost", "count": "n_findings"})
    .sort_values("total_cost", ascending=False)
)
by_type["label"] = by_type["waste_type"].map(_TYPE_LABELS)
by_type["total_cost"] = by_type["total_cost"].round(2)

print("Waste by type:")
print(by_type[["label", "total_cost", "n_findings"]].to_string(index=False))

In [ ]:
_TYPE_COLORS = {
    "unused_commitment":        "#ef4444",
    "idle_compute":             "#f97316",
    "zombie_resource":          "#8b5cf6",
    "underutilized_commitment": "#eab308",
}

by_team_type = (
    findings_df.groupby(["allocated_team", "waste_type"])["cost_impact"]
    .sum()
    .reset_index()
)

fig = px.bar(
    by_team_type,
    x="allocated_team", y="cost_impact", color="waste_type",
    color_discrete_map=_TYPE_COLORS,
    title="Waste by Team and Type",
    labels={"cost_impact": "Cost Impact (USD)", "allocated_team": "Team", "waste_type": "Waste Type"},
    category_orders={"waste_type": list(_TYPE_COLORS.keys())},
)
fig.update_layout(height=380, legend=dict(orientation="h", y=1.12))
fig.show()

## Cell 5 — Causal Reasoning

In [ ]:
from intelligence.causal_engine import run as explain

insights = explain(df, findings)

print(f"Causal insights: {len(insights)}\n")

for ins in insights:
    team   = ins["scope"].replace("team:", "")
    chg    = ins["cost_change_pct"]
    arrow  = "▲" if chg > 0 else ("▼" if chg < 0 else "→")
    anomaly_tag = " [ANOMALY]" if ins["anomaly"] else ""
    print(f"{'─'*60}")
    print(f"Team : {team}   {arrow}{abs(chg):.1f}% MoM   "
          f"confidence={ins['confidence']:.2f}{anomaly_tag}")
    print(f"Period: {ins['period']}")
    for rc in ins["root_causes"]:
        print(f"  [{rc['weight']:.2f}] {rc['cause']}")
        print(f"         {rc['evidence']}")
print(f"{'─'*60}")

## Cell 6 — Cost Trend Visualisation

In [ ]:
from intelligence.causal_engine import build_nec_trend, compute_zscore_anomaly

trend_df = build_nec_trend(df)
trend_df = compute_zscore_anomaly(trend_df)

print(f"Trend rows: {len(trend_df)}")
trend_df.sort_values(["allocated_team", "billing_month"]).head(15)

In [ ]:
_TEAM_COLORS = ["#f97316", "#2563eb", "#22c55e", "#8b5cf6", "#eab308", "#ef4444"]
_TEAM_LABELS = {
    "data-eng": "Data Engineering", "platform": "Platform",
    "frontend": "Frontend",         "backend":  "Backend",
    "ml":       "Machine Learning",
}

n_months = trend_df["billing_month"].nunique()

if n_months < 2:
    # Single month — bar chart
    bar = (
        trend_df.groupby("allocated_team")["period_nec"]
        .sum().reset_index().sort_values("period_nec", ascending=True)
    )
    bar["label"] = bar["allocated_team"].map(_TEAM_LABELS).fillna(bar["allocated_team"])
    fig = px.bar(
        bar, x="period_nec", y="label", orientation="h",
        color="label", color_discrete_sequence=_TEAM_COLORS,
        title=f"NEC by Team ({trend_df['billing_month'].max()})",
        labels={"period_nec": "NEC Used (USD)", "label": "Team"},
    )
    fig.update_layout(height=320, showlegend=False)
    fig.show()
else:
    # Multi-month line chart with anomaly overlay
    fig = go.Figure()
    for i, team in enumerate(sorted(trend_df["allocated_team"].unique())):
        t = trend_df[trend_df["allocated_team"] == team].sort_values("billing_month")
        color = _TEAM_COLORS[i % len(_TEAM_COLORS)]
        label = _TEAM_LABELS.get(team, team.replace("-", " ").title())
        fig.add_trace(go.Scatter(
            x=t["billing_month"].astype(str), y=t["period_nec"],
            mode="lines+markers", name=label,
            line=dict(color=color, width=2), marker=dict(size=6),
        ))
        anomalies = t[t["anomaly"] == True]
        if not anomalies.empty:
            fig.add_trace(go.Scatter(
                x=anomalies["billing_month"].astype(str), y=anomalies["period_nec"],
                mode="markers", name=f"{label} anomaly",
                marker=dict(color="#ef4444", size=14, symbol="circle-open",
                            line=dict(width=2)),
                showlegend=False,
            ))
    fig.update_layout(
        title="NEC Trend by Team (MoM) — red circles = statistical anomalies",
        xaxis_title="Billing Month", yaxis_title="NEC Used (USD)",
        height=420, hovermode="x unified",
        legend=dict(orientation="h", y=1.12),
    )
    fig.show()

## Cell 7 — Impact Simulation

In [ ]:
from intelligence.impact_simulator import run as simulate

recs = simulate(findings)

print(f"Recommendations: {len(recs)}")
print()

recs_df = pd.DataFrame(recs)
recs_df[["allocated_team", "action", "current_cost",
         "estimated_savings", "savings_pct", "risk", "priority_score"]].head(10)

In [ ]:
_ACTION_COLORS = {
    "release_commitment": "#2563eb",
    "resize_down":        "#f97316",
    "remove_resource":    "#8b5cf6",
}
_RISK_COLORS = {"Low": "#22c55e", "Medium": "#eab308", "High": "#ef4444"}

by_action = (
    recs_df.groupby("action")["estimated_savings"]
    .sum().reset_index().sort_values("estimated_savings", ascending=True)
)

fig = px.bar(
    by_action, x="estimated_savings", y="action", orientation="h",
    color="action", color_discrete_map=_ACTION_COLORS,
    title="Estimated Savings by Action Type",
    labels={"estimated_savings": "Savings (USD)", "action": "Action"},
    text=by_action["estimated_savings"].map("${:,.2f}".format),
)
fig.update_traces(textposition="outside")
fig.update_layout(height=260, showlegend=False, margin=dict(r=100))
fig.show()

In [ ]:
# Top 10 recommendations table — sorted by priority_score
top10 = (
    recs_df
    .sort_values("priority_score", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top10.index += 1  # 1-based rank
top10[["allocated_team", "action", "current_cost",
       "estimated_savings", "savings_pct", "risk", "priority_score"]]

## Cell 8 — Executive Summary

In [ ]:
# ── Totals ─────────────────────────────────────────────────────────────────
total_nec_used   = df["nec_used"].sum()
total_nec_waste  = df["nec_waste"].sum()
detected_waste   = findings_df["cost_impact"].sum()
total_savings    = recs_df["estimated_savings"].sum()
quick_win_savings = recs_df.loc[recs_df["risk"] == "Low", "estimated_savings"].sum()
n_anomalies      = sum(1 for i in insights if i["anomaly"])
n_quick_wins     = (recs_df["risk"] == "Low").sum()

months_loaded    = sorted(df["billing_month"].unique())

print("══════════════════════════════════════════════════════")
print("  COST INTELLIGENCE SUMMARY")
print("══════════════════════════════════════════════════════")
print(f"  Billing months   : {', '.join(str(m) for m in months_loaded)}")
print(f"  Total NEC (used) : ${total_nec_used:>12,.2f}")
print(f"  Total NEC (waste): ${total_nec_waste:>12,.2f}")
print()
print(f"  Waste findings   : {len(findings)} ({detected_waste:,.2f} USD detected)")
print(f"  Causal insights  : {len(insights)} teams analysed")
print(f"  Anomalies        : {n_anomalies}")
print()
print(f"  Recommendations  : {len(recs)} actions")
print(f"  Total savings    : ${total_savings:>12,.2f}/month (if all acted on)")
print(f"  Quick wins (Low) : ${quick_win_savings:>12,.2f}/month ({n_quick_wins} actions, no penalty)")
print(f"  Savings as % NEC : {total_savings / total_nec_used * 100:.2f}%")
print()

# Top recommendation
top = recs_df.iloc[0]
print(f"  TOP RECOMMENDATION:")
print(f"    Team   : {top['allocated_team']}")
print(f"    Action : {top['action']}")
print(f"    Savings: ${top['estimated_savings']:,.2f}/month ({top['savings_pct']:.1f}%)")
print(f"    Risk   : {top['risk']}")
print("══════════════════════════════════════════════════════")